## Redistribuicao etre cervejas produzidas no neno sem considerar araguaia 

In [4]:
# ── Imports principais ─────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualização em notebook (Jupyter / VSCode) ────────────
from IPython.display import display

# ── Configuração opcional do pandas ────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [5]:
import pandas as pd
# Produção por cerveja/fábrica/semana (valores da foto)
producao = {
    "patagonia": {"CE": {"W0": 12240, "W1": 1800,  "W2": 5040,  "W3": 12600},
                  "PE": {"W0": 0,     "W1": 0,      "W2": 0,     "W3": 0}},
    "malzbier ":  {"CE": {"W0": 0,     "W1": 9000,   "W2": 7560,  "W3": 0},
                  "PE": {"W0": 16200, "W1": 0,      "W2": 12960, "W3": 0}},
    "colorado ":  {"CE": {"W0": 0,     "W1": 0,      "W2": 0,     "W3": 0},
                  "PE": {"W0": 5400,  "W1": 0,      "W2": 10800, "W3": 0}},
    "goose":     {"CE": {"W0": 0,     "W1": 0,      "W2": 0,     "W3": 0},
                  "PE": {"W0": 5400,  "W1": 14400,  "W2": 0,     "W3": 12600}},
}

def analisar_cerveja(cerveja, df_all, producao):
    df_cerv = df_all[df_all["Cerveja"] == cerveja].copy()

    ei_w0 = df_cerv[df_cerv["Semana"] == "W0"].set_index("Regiao")["EI_Semana"].to_dict()

    semanas = ["W0", "W1", "W2", "W3"]
    regioes = sorted(df_cerv["Regiao"].unique().tolist())

    dem_dict = {}
    for sem in semanas:
        dem_dict[sem] = df_cerv[df_cerv["Semana"] == sem].set_index("Regiao")["Demanda"].to_dict()

    estoque = {r: float(ei_w0.get(r, 0) or 0) for r in regioes}
    resultados = []

    for i, sem in enumerate(semanas):
        prod_ce   = producao[cerveja]["CE"].get(sem, 0)
        prod_pe   = producao[cerveja]["PE"].get(sem, 0)
        prod_disp = prod_ce + prod_pe
        prox_sem  = semanas[i + 1] if i + 1 < len(semanas) else None

        nec = {}
        for r in regioes:
            dem     = dem_dict[sem].get(r, 0)
            ei      = estoque[r]
            dem_prox = dem_dict[prox_sem].get(r, 0) if prox_sem else dem
            ef_alvo  = 12 * (dem_prox / 6)
            envio_nec = max(0, ef_alvo + dem - ei)
            nec[r] = {"dem": dem, "ei": ei, "ef_alvo": ef_alvo,
                      "envio_nec": envio_nec, "dem_prox": dem_prox}

        total_nec = sum(v["envio_nec"] for v in nec.values())

        for r in regioes:
            dem       = nec[r]["dem"]
            ei        = nec[r]["ei"]
            ef_alvo   = nec[r]["ef_alvo"]
            envio_nec = nec[r]["envio_nec"]
            dem_prox  = nec[r]["dem_prox"]

            envio_prod = prod_disp * (envio_nec / total_nec) if total_nec > 0 else 0
            vem_sp     = max(0, envio_nec - envio_prod)

            if sem in ["W0", "W1", "W2"]:
                modal         = "Rodoviário"
                vol_enviar_sp = vem_sp / 0.95
            else:
                modal         = "Cabotagem"
                vol_enviar_sp = vem_sp

            ef    = ei + envio_prod + vem_sp - dem
            suf_f = ef / (dem_prox / 6) if dem_prox > 0 else 0

            resultados.append({
                "Cerveja":       cerveja,
                "Semana":        sem,
                "Regiao":        r,
                "Demanda":       round(dem, 2),
                "Dem_Proxima":   round(dem_prox, 2),
                "EI":            round(ei, 2),
                "Prod_NE":       round(envio_prod, 2),
                "Vem_SP":        round(vem_sp, 2),
                "Vol_Enviar_SP": round(vol_enviar_sp, 2),
                "Modal_SP":      modal if vem_sp > 0 else "-",
                "EF":            round(ef, 2),
                "Suf_f":         round(suf_f, 1),
            })

            estoque[r] = ef

    return pd.DataFrame(resultados)

# ── Roda para todas as cervejas ───────────────────────────────────────────────
cervejas = ["patagonia", "malzbier ", "colorado ", "goose"]
df_final = pd.concat([analisar_cerveja(c, df_all, producao) for c in cervejas], ignore_index=True)

# ── Exibe por cerveja e semana ────────────────────────────────────────────────
for cerveja, df_c in df_final.groupby("Cerveja"):
    print(f"\n{'#'*60}  {cerveja.upper()}  {'#'*60}")
    for sem, grupo in df_c.groupby("Semana"):
        print(f"\n{'='*50}  {sem}  {'='*50}")
        display(grupo.reset_index(drop=True))

# ── Resumo SP ─────────────────────────────────────────────────────────────────
print("\n=== RESUMO — O QUE VEM DE SP ===")
sp = df_final[df_final["Vem_SP"] > 0][["Cerveja","Semana","Regiao","Demanda","Prod_NE","Vem_SP","Vol_Enviar_SP","Modal_SP","Suf_f"]]
display(sp if len(sp) > 0 else pd.DataFrame([{"Resultado": "Nenhuma transferência necessária!"}]))

# ── Produção total por cerveja/semana ─────────────────────────────────────────
print("\n=== PRODUÇÃO TOTAL NE POR CERVEJA/SEMANA ===")
display(df_final.groupby(["Cerveja","Semana"])["Prod_NE"].sum().reset_index())

NameError: name 'df_all' is not defined